# OpenClaw Overall Harness Lab

This Colab notebook uses OpenClaw itself to make harness surfaces visible.

A raw LLM call receives explicit input and returns text. A harnessed agent turn surrounds the model with runtime-managed context, workspace access, tool surface, policy, session state, and observable output.

There are two optional execution lanes:

1. **DeepSeek API lane**: stable live demo path, but it requires a student-owned API key and may incur API cost.
2. **Ollama lane**: local-model path inside Colab, but it depends on Colab CPU/GPU/RAM availability and model size.

You can replace the provider or model if you have configured another backend. DeepSeek and `llama3.2:3b` are defaults for this lab, not requirements.

## 0. Runtime notes

This notebook is designed for Google Colab. It does not modify your local machine.

The DeepSeek lane is optional. Run it only if you have a `DEEPSEEK_API_KEY` and accept the API cost. You may use another configured API provider instead.

The Ollama lane is also optional. It downloads and serves a small model inside the Colab runtime, so it needs enough runtime resources. You may replace the Ollama model with a stronger or smaller one. A self-hosted backend such as vLLM or SGLang can also work if your OpenClaw setup exposes it as a configured model route.

For Ollama, GPU is recommended for the full agent turn. CPU may be impractically slow. Small local models may pass the model probe but stall, miss the workspace marker, or produce odd output during a full tool-using agent turn; that is an observation about workload and model/backend capacity.

Important: printed notebook output is for humans. A previous cell may print `notes.txt`, but that text is not automatically passed into `openclaw infer model run`. This notebook uses an A/B/C comparison: raw infer without `notes.txt`, raw infer with `notes.txt` explicitly pasted into the prompt, and `agent --local` where the runtime mediates workspace access.

In [ ]:
import os
import pathlib
import shutil
import shlex
import subprocess

REPO_URL = "https://github.com/SleepyLGod/openclaw-teaching.git"
REPO_DIR = pathlib.Path("/content/openclaw-teaching")
LAB_DIR = REPO_DIR / "labs" / "overall-harness"
WORKSPACE_SRC = LAB_DIR / "workspace"
OPENCLAW_WORKSPACE = pathlib.Path.home() / ".openclaw" / "workspace"

def section(title: str) -> None:
    line = "=" * len(title)
    print(f"\n{line}\n{title}\n{line}")

def show_file(path: pathlib.Path) -> None:
    print(f"\n--- {path} ---")
    print(path.read_text(encoding="utf-8"))

def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f"\n$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return proc


## 1. Clone the teaching repo

The GitBook page explains the ideas. The repo contains the runnable lab files.

In [ ]:
section("1. Clone the teaching repo")
print("Observation target: load the lab materials that define the OpenClaw workspace.")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
_ = run(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")

print("Lab directory:", LAB_DIR)
print("Workspace source directory:", WORKSPACE_SRC)
print("Workspace files and contents before copying into OpenClaw:")
for path in sorted(WORKSPACE_SRC.iterdir()):
    if path.is_file():
        show_file(path)


## 2. Install OpenClaw

The installer handles Node and OpenClaw setup. We skip interactive onboarding because the notebook configures providers explicitly later.

In [ ]:
section("2. Install OpenClaw")
print("Observation target: the CLI exists before we test any model or full agent turn.")

_ = run("curl -fsSL https://openclaw.ai/install.sh | bash -s -- --no-onboard", check=False)

# Make common install locations visible to this notebook process.
os.environ["PATH"] = ":".join([
    str(pathlib.Path.home() / ".local" / "bin"),
    str(pathlib.Path.home() / ".openclaw" / "bin"),
    os.environ.get("PATH", ""),
])

_ = run("command -v openclaw || true", check=False)
_ = run("openclaw --version", check=False)
_ = run("openclaw doctor", check=False)


## 3. Harness inputs: prepare the OpenClaw workspace

These files are intentionally small. They make context and workspace surfaces visible.

`AGENTS.md`, `SOUL.md`, and `USER.md` are bootstrap/context artifacts. `notes.txt` is a controlled workspace fixture with a marker used for observation.

In [ ]:
section("3. Prepare the OpenClaw workspace")
print("Observation target: workspace/bootstrap files become runtime inputs for the full agent turn.")

OPENCLAW_WORKSPACE.mkdir(parents=True, exist_ok=True)
for src in WORKSPACE_SRC.iterdir():
    if src.is_file():
        shutil.copy2(src, OPENCLAW_WORKSPACE / src.name)

print("OpenClaw workspace target:", OPENCLAW_WORKSPACE)
print("Copied bootstrap/workspace files:")
for path in sorted(OPENCLAW_WORKSPACE.glob("*")):
    if path.is_file() and path.name in {"SOUL.md", "USER.md", "AGENTS.md", "notes.txt"}:
        show_file(path)


## 4. DeepSeek API lane

This is the stable demo path, but it is not required. Put your API key into Colab Secrets or paste it when prompted only if you have your own DeepSeek account and accept the API cost.

The provider is `deepseek`; the default model for this lab is `deepseek/deepseek-v4-flash`. You may replace it with another configured API model if your environment supports it.

In [ ]:
section("4. DeepSeek API provider setup")
print("Observation target: configure the stable API provider without printing the API key.")

import getpass

if not os.environ.get("DEEPSEEK_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        secret = None

    if secret:
        os.environ["DEEPSEEK_API_KEY"] = secret
        print("Loaded DEEPSEEK_API_KEY from Colab Secrets.")
    else:
        os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Enter DEEPSEEK_API_KEY: ")
else:
    print("DEEPSEEK_API_KEY already exists in the environment.")

_ = run('openclaw onboard --non-interactive --mode local --auth-choice deepseek-api-key --deepseek-api-key "$DEEPSEEK_API_KEY" --skip-health --accept-risk', check=False)
_ = run("openclaw models list --all --provider deepseek", check=False)


### 4A. Model route: DeepSeek smoke probe

This checks model-route/auth health with a tiny prompt. It is still not a full agent turn.


In [ ]:
section("4A. DeepSeek model smoke probe")
DEEPSEEK_MODEL = "deepseek/deepseek-v4-flash"
print("Provider: deepseek")
print("Model reference:", DEEPSEEK_MODEL)
print("Expected observation: this checks the selected model route only, not the full agent turn.")
_ = run(f"openclaw infer model run --local --model {DEEPSEEK_MODEL} --prompt 'Reply exactly: COURSE_OK' --json", check=False)


### 4B. Raw LLM call A: no workspace context

This is the negative control. The semantic task is about `notes.txt`, but the raw LLM call only receives the prompt below. It does not receive the file content.

Target observation: the model route works, but the model should not know `HARNESS_CONTEXT_VISIBLE` unless it guesses or hallucinates. This proves only that raw infer uses explicit prompt input.


In [ ]:
section("4B. DeepSeek raw infer A: no workspace context")
raw_no_context_prompt = "What is the workspace marker in notes.txt? If the content of notes.txt is not included in this prompt, reply exactly: NO_WORKSPACE."
print("Provider: deepseek")
print("Model reference:", DEEPSEEK_MODEL)
print("Expected observation: raw infer receives only the prompt string, not notes.txt.")
_ = run(f"openclaw infer model run --local --model {DEEPSEEK_MODEL} --prompt {shlex.quote(raw_no_context_prompt)} --json", check=False)


### 4C. Raw LLM call B: explicit `notes.txt` context

This is the positive control. We paste the content of `notes.txt` into the raw infer prompt ourselves.

Target observation: raw infer can now answer the marker. That means the comparison is not about model intelligence. It is about who constructs the input: you, manually, or the agent runtime.


In [ ]:
section("4C. DeepSeek raw infer B: explicit notes.txt context")
notes_text = (OPENCLAW_WORKSPACE / "notes.txt").read_text(encoding="utf-8")
raw_explicit_context_prompt = f"""You are given the exact content of notes.txt:

<notes.txt>
{notes_text}
</notes.txt>

Answer in one sentence: what is the workspace marker, and what does this lab teach?"""
print("Provider: deepseek")
print("Model reference:", DEEPSEEK_MODEL)
print("Expected observation: this is still raw infer, but the marker is now explicit prompt context.")
_ = run(f"openclaw infer model run --local --model {DEEPSEEK_MODEL} --prompt {shlex.quote(raw_explicit_context_prompt)} --json", check=False)


### 4D. Harnessed agent turn: DeepSeek

This is the runtime-mediated path. The user message asks the agent to read `notes.txt`, but the notebook does not paste the file content into the prompt.

Watch for three surfaces in the output: workspace/context behavior from `notes.txt`, policy/action-boundary diagnostics such as `tool-policy`, and session isolation from the fresh `--session-key`.


In [ ]:
section("4D. DeepSeek harnessed agent turn")
print("This is now a full runtime/harness turn.")
print("Session key: harness-deepseek-a")
print("Workspace file expected to matter: notes.txt")
print("Target observation: unlike 4B, this turn asks the runtime-mediated agent path to obtain or reference the workspace marker; compare your actual output.")
_ = run("openclaw agent --local --session-key harness-deepseek-a --message 'Read notes.txt and tell me the workspace marker, then answer in one sentence: what does this lab teach?'", check=False)

print("")
print("This second full turn uses a fresh session key to observe bootstrap instructions without stale history.")
print("Session key: harness-deepseek-b")
print("Workspace files expected to matter: USER.md and SOUL.md")
_ = run("openclaw agent --local --session-key harness-deepseek-b --message 'How should you address me, and what style should you use?'", check=False)


## 5. Ollama lane

This is the local-model path inside Colab. It may be slower and less reliable than the API lane, especially on CPU-only runtimes.

The harness surfaces are the same as the DeepSeek lane. The backend is different: a local model server instead of a hosted API route. You may replace the model with a stronger or smaller Ollama model depending on your available resources.

In [ ]:
section("5. Install and start Ollama")
print("Observation target: install Ollama, start its local API server, and register local-only auth for OpenClaw.")

# Ollama's current Linux installer requires zstd on Colab's Ubuntu image.
_ = run("apt-get update && apt-get install -y zstd curl ca-certificates", check=False)
_ = run("curl -fsSL https://ollama.com/install.sh | sh", check=False)

# OpenClaw requires an Ollama API key value to register the provider. For local Ollama, any value works.
os.environ["OLLAMA_API_KEY"] = "ollama-local"
print("Set OLLAMA_API_KEY=ollama-local for local Ollama provider registration.")

# Start Ollama in the background if it is not already running.
_ = run("pgrep -x ollama >/dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)", check=False)
_ = run("sleep 5 && curl -s http://127.0.0.1:11434/api/tags || true", check=False)


### 5A. Pull a small model

`llama3.2:3b` is small enough for a Colab demo, but it is still resource-dependent. You can replace it with a stronger model if your runtime has enough GPU/RAM, or a smaller model if you only want to observe the raw LLM probe.

On a Colab T4, a quantized 7B or 8B Ollama model is a reasonable next experiment. A 14B model may fit in some cases, but the full harnessed turn can become slow or memory constrained because it carries more context than the smoke probe. Avoid making a larger local model the default classroom path unless you have tested it on the exact runtime.

In [ ]:
section("5A. Pull a small Ollama model")

# To try another Ollama model, change only this value, then rerun cells 5A-5D.
# Do not add --model to the agent command; this cell sets the OpenClaw default model.
OLLAMA_MODEL_NAME = "llama3.2:3b"
OLLAMA_MODEL_REF = f"ollama/{OLLAMA_MODEL_NAME}"

print("Provider: ollama")
print("Model reference:", OLLAMA_MODEL_REF)
print("Expected observation: model download succeeds, OpenClaw inspects the Ollama provider, and the model becomes the default.")
_ = run(f"ollama pull {OLLAMA_MODEL_NAME}", check=False)
_ = run("openclaw models list --all --provider ollama", check=False)
_ = run(f"openclaw models set {OLLAMA_MODEL_REF}", check=False)


### 5B. Model route: Ollama smoke probe

This checks local Ollama inference before the A/B/C comparison.

Exact output is ideal, but a small model may return a close string such as `OLLambda_OK`. That still shows the model route is alive; the mismatch is a model-quality signal, not a provider setup failure.


In [ ]:
section("5B. Ollama model smoke probe")
print("Provider: ollama")
print("Model reference:", OLLAMA_MODEL_REF)
print("Expected observation: this checks local model inference before trying a full agent turn.")
print("If the token is close but not exact, the route is alive but instruction following is weak.")
_ = run(f"openclaw infer model run --local --model {OLLAMA_MODEL_REF} --prompt 'Reply exactly: OLLAMA_OK' --json", check=False)


### 5C. Raw LLM call A: no workspace context

This repeats the negative control with a local model. It asks about the workspace marker through `infer model run`, but the prompt does not include `notes.txt`.

The marker was printed earlier for you, but printed notebook output is not automatically passed into this command.


In [ ]:
section("5C. Ollama raw infer A: no workspace context")
ollama_raw_no_context_prompt = "What is the workspace marker in notes.txt? If the content of notes.txt is not included in this prompt, reply exactly: NO_WORKSPACE."
print("Provider: ollama")
print("Model reference:", OLLAMA_MODEL_REF)
print("Expected observation: raw infer receives only the prompt string, not notes.txt.")
_ = run(f"openclaw infer model run --local --model {OLLAMA_MODEL_REF} --prompt {shlex.quote(ollama_raw_no_context_prompt)} --json", check=False)


### 5D. Raw LLM call B: explicit `notes.txt` context

This repeats the positive control with a local model. We paste `notes.txt` into the raw infer prompt ourselves.

If a small model still misses the marker or gives a strange answer, treat that as a model-quality/context-following observation, not as a provider setup failure.


In [ ]:
section("5D. Ollama raw infer B: explicit notes.txt context")
notes_text = (OPENCLAW_WORKSPACE / "notes.txt").read_text(encoding="utf-8")
ollama_raw_explicit_context_prompt = f"""You are given the exact content of notes.txt:

<notes.txt>
{notes_text}
</notes.txt>

Answer in one sentence: what is the workspace marker, and what does this lab teach?"""
print("Provider: ollama")
print("Model reference:", OLLAMA_MODEL_REF)
print("Expected observation: this is still raw infer, but the marker is now explicit prompt context.")
_ = run(f"openclaw infer model run --local --model {OLLAMA_MODEL_REF} --prompt {shlex.quote(ollama_raw_explicit_context_prompt)} --json", check=False)


### 5E. Harnessed agent turn: Ollama

This is a workload observation, not a required success condition. Small local models can return the marker, miss the marker, stall, or emit strange code-like text when the full agent context and tool surface are present.

GPU is recommended. On CPU, this cell can take a very long time because the harnessed turn sends much more context than the probe. If the model probe passed but this full agent turn misses the marker, gives off-task output, or is too slow, map the symptom to a harness surface: model route, context budget, tool surface, or runtime capacity.


In [ ]:
section("5E. Ollama harnessed agent turn")
print("This is now a full runtime/harness turn with a small local model.")
print("Session key: harness-ollama-a")
print("Workspace file expected to matter: notes.txt")
print("The model was set as OpenClaw's default, so this command intentionally does not use --model override.")
print("Expected observation: this exercises the same harnessed turn under a small local model.")
print("Compare what happened: the output may return the marker, miss it, emit tool-like text, or become slow.")
_ = run("openclaw agent --local --session-key harness-ollama-a --message 'Read notes.txt and tell me the workspace marker, then answer in one sentence: what does this lab teach?'", check=False)


## 6. Observation -> Harness Surface

Use this worksheet after running or reading the notebook output.

| Observation | Harness surface |
| --- | --- |
| `infer model run` smoke probe succeeds | Model route |
| Raw infer A does not know `HARNESS_CONTEXT_VISIBLE` | Explicit input / negative control |
| Raw infer B answers after `notes.txt` is pasted into the prompt | Explicit context construction / positive control |
| `agent --local` target observation: obtains or references the marker without manual prompt stuffing | Workspace surface / runtime mediation / action boundary |
| Output includes `tool-policy` | Policy surface / action boundary |
| Fresh `--session-key` changes what history is available | Session surface |
| Ollama smoke output is close but not exact | Model route works; instruction following is weak |
| Ollama full turn is slow, misses the marker, or emits odd output | Model route / context budget / runtime capacity |

Questions:

1. Which command was only a model-route smoke probe?
2. In raw infer A, what exact input did the model receive?
3. In raw infer B, why can the same raw model path answer the marker?
4. In the harnessed agent turn, what did the notebook not paste manually?
5. What did the tool-policy line show about the action boundary?
6. Which observation maps to context, workspace, policy, session, or model route?
7. Pick one changed or surprising output. Which harness surface would you inspect or improve first, and what smallest change would you make?
8. Which surfaces should be saved for deeper Guardrails, Eval, Multi-Agent, or Skills chapters?


## 7. Key takeaway

An LLM call returns text from explicit input. A harnessed agent turn runs inside a system boundary.

Harness Engineering is the practice of shaping that boundary: what the model sees, what actions are mediated, how workflow state is tracked, how behavior is observed, and where the system should be improved when behavior changes.